# 00 Environment and Dataset Audit

Sources: Kaggle Amazon Reviews (fine-grained), Hugging Face GoEmotions (simplified)
Expected outputs: Hardware inspection, workspace setup, and dataset audit manifests.

In [1]:
import os, sys, platform, subprocess, json
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA runtime:", torch.version.cuda)
        print("GPU memory (GB):", round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
        ))
except Exception as exc:
    print("PyTorch inspection failed:", repr(exc))

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("nvidia-smi unavailable:", repr(exc))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Working directory: /content
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA runtime: 13.0
GPU memory (GB): 14.56
Fri Nov 13 16:48:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/NLP_Lab')
for name in [
    'notebooks', 'raw_data/kaggle_amazon',
    'raw_data/huggingface_go_emotions',
    'processed_data/amazon_reviews', 'processed_data/go_emotions',
    'checkpoints/word2vec_bilstm', 'checkpoints/distilbert_goemotions',
    'final_models', 'logs', 'reports/figures', 'reports/metrics',
    'reports/failures', 'exports'
]:
    (ROOT / name).mkdir(parents=True, exist_ok=True)

# Verification
if ROOT.exists():
    print(f"Workspace verified at: {ROOT}")
    # Write temporary file to verify write access
    test_file = ROOT / "write_test.tmp"
    test_file.write_text("ok")
    if test_file.exists():
        print("Write access confirmed.")
        test_file.unlink()
else:
    print("Workspace creation failed.")

Mounted at /content/drive
Workspace verified at: /content/drive/MyDrive/NLP_Lab
Write access confirmed.


In [3]:
# Reinstall datasets, pandas, and pyarrow to mutually satisfy Google Colab, RAPIDS, and HF Datasets
!pip install -q "pandas==2.2.3" "pyarrow>=19.0.0,<24.0.0" "datasets<5.0.0"

In [4]:
# Verify the package versions now
import pandas as pd
import pyarrow as pa
print("Pandas version:", pd.__version__)
print("PyArrow version:", pa.__version__)

Pandas version: 2.2.3
PyArrow version: 23.0.1


In [5]:
import importlib.metadata as md

packages = ["torch", "numpy", "pandas", "transformers", "datasets", "gensim", "scikit-learn"]
print("--- Before Install ---")
for p in packages:
    try:
        print(f"{p}: {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p}: NOT INSTALLED")

# Install required packages quietly
!pip install -q -U "gensim" "datasets" "transformers" "accelerate" \
  "evaluate" "scikit-learn" "pandas" "pyarrow" "matplotlib" \
  "seaborn" "safetensors" "huggingface_hub" "kagglehub" \
  "onnx" "onnxruntime"

print("\n--- After Install ---")
for p in packages + ["accelerate", "onnx", "kagglehub"]:
    try:
        print(f"{p}: {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p}: NOT INSTALLED")

--- Before Install ---
torch: 2.14.1+cu130
numpy: 2.4.6
pandas: 2.2.3
transformers: 5.22.1
datasets: 4.8.5
gensim: NOT INSTALLED
scikit-learn: 1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.

--- After Install ---
torch: 2.14.1+cu130
numpy: 2.4.6
pandas: 3.1.1
transformers: 5.23.0
datasets: 5.2.0
gensim: 4.4.0
scikit-learn: 1.9.1
accelerate: 1.15.0
onnx: 1.23.2
kagglehub: 1.0.3


In [6]:
import json, os, sys, platform, torch
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/NLP_Lab')
env_data = {
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None
}

env_path = ROOT / "reports" / "environment.json"
req_path = ROOT / "reports" / "environment_requirements.txt"

with open(env_path, "w") as f:
    json.dump(env_data, f, indent=2)

os.system(f"pip freeze > {req_path}")

print(f"Environment manifest saved: {env_path.exists()}")
print(f"Requirements saved: {req_path.exists()}")

Environment manifest saved: True
Requirements saved: True


In [7]:
import kagglehub
import shutil
from pathlib import Path

RAW_DIR = Path('/content/drive/MyDrive/NLP_Lab/raw_data/kaggle_amazon')

try:
    print("Attempting to download dataset via kagglehub...")
    path = kagglehub.dataset_download("yacharki/amazon-reviews-for-sentianalysis-finegrained-csv")
    print(f"Downloaded to cache: {path}")

    # Move CSVs to Drive
    for item in Path(path).glob('*.csv'):
        dest = RAW_DIR / item.name
        if not dest.exists():
            shutil.copy2(item, dest)
            print(f"Copied {item.name} to Drive.")
        else:
            print(f"{item.name} already exists in Drive.")

except Exception as e:
    print(f"Automated download failed (likely missing credentials): {repr(e)}")
    print("Please download manually from Kaggle and upload the CSV to MyDrive/NLP_Lab/raw_data/kaggle_amazon/")

print("\nCurrent files in Kaggle raw directory:")
for p in RAW_DIR.rglob('*'):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size / 1024**2:.2f} MB)")

Attempting to download dataset via kagglehub...


100%|██████████| 624M/624M [00:08<00:00, 75.6MB/s]

Extracting files...


Downloaded to cache: /root/.cache/kagglehub/datasets/yacharki/amazon-reviews-for-sentianalysis-finegrained-csv/versions/2

Current files in Kaggle raw directory:
- test.csv (276.26 MB)
- train.csv (1273.86 MB)


In [8]:
import pandas as pd
from pathlib import Path
import json

RAW_DIR = Path('/content/drive/MyDrive/NLP_Lab/raw_data/kaggle_amazon')
csv_files = list(RAW_DIR.rglob('*.csv'))
print("Found CSVs:", [f.name for f in csv_files])

schema_info = {}

for csv_path in csv_files:
    print(f"\n--- Inspecting {csv_path.name} ---")
    try:
        # Read just a few rows to get the schema without blowing up RAM
        preview = pd.read_csv(csv_path, nrows=5)
        cols = preview.columns.tolist()
        print("Columns:", cols)
        print("Data Types:\n", preview.dtypes)
        print("Preview:\n", preview.head(2).to_string())

        schema_info[csv_path.name] = {
            "columns": cols,
            "dtypes": preview.dtypes.astype(str).to_dict()
        }
    except Exception as e:
        print(f"Failed to read {csv_path.name}: {repr(e)}")

# Save Amazon schema
schema_path = Path('/content/drive/MyDrive/NLP_Lab/processed_data/amazon_reviews/amazon_schema.json')
with open(schema_path, "w") as f:
    json.dump(schema_info, f, indent=2)
print(f"\nAmazon schema saved: {schema_path.exists()}")

Found CSVs: ['test.csv', 'train.csv']

--- Inspecting test.csv ---
Columns: ['class_index', 'review_title', 'review_text']
Data Types:
 class_index      int64
review_title    object
review_text     object
dtype: object
Preview:
    class_index             review_title                                                                                                                                                                                                                                                                                                                                                                                                   review_text
0            1          mens ultrasheer  This model may be ok for sedentary types, but I'm active and get around alot in my job - consistently found these stockings rolled up down by my ankles! Not Good!! Solution: go with the standard compression stocking, 20-30, stock #114622. Excellent support, stays up and gives me what I need.

In [2]:
from datasets import load_dataset
import json
from pathlib import Path

print("--- Loading GoEmotions (Simplified) ---")
try:
    emotion_ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    print(emotion_ds)

    features = emotion_ds["train"].features
    print("\nFeatures:\n", features)
    print("\nSample [0]:\n", emotion_ds["train"][0])

    # Extract label names if available
    labels = features['labels'].feature.names if hasattr(features['labels'], 'feature') else "Unknown"
    print(f"\nLabel count: {len(labels) if isinstance(labels, list) else 'Unknown'}")

    # Save GoEmotions schema
    goemotions_schema = {
        "splits": {k: len(v) for k, v in emotion_ds.items()},
        "features": {k: str(v) for k, v in features.items()}
    }

    schema_path = Path('/content/drive/MyDrive/NLP_Lab/processed_data/go_emotions/goemotions_schema.json')
    with open(schema_path, 'w') as f:
        json.dump(goemotions_schema, f, indent=2)

    print(f"\nGoEmotions schema saved: {schema_path.exists()}")

except Exception as e:
    if "pyarrow.lib.Array size changed" in str(e):
        print(f"\n[!] BINARY INCOMPATIBILITY DETECTED: {repr(e)}")
        print("To fix this, please restart your Google Colab session: Menu -> Runtime -> Restart session.")
    else:
        print(f"Failed to load GoEmotions: {repr(e)}")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


--- Loading GoEmotions (Simplified) ---


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

... (3 outputs left out)

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

Features:
 {'text': Value('string'), 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])), 'id': Value('string')}

Sample [0]:
 {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}

Label count: 28

GoEmotions schema saved: True


In [3]:
import kagglehub
import shutil
import pandas as pd
import json
from pathlib import Path

RAW_DIR = Path('/content/drive/MyDrive/NLP_Lab/raw_data/kaggle_amazon')
print("--- Applying Fix: Locating Kaggle files ---")

# Get cache path (instantaneous if already downloaded)
path = kagglehub.dataset_download("yacharki/amazon-reviews-for-sentianalysis-finegrained-csv")

# Use rglob to search recursively through the cache
csvs_found = list(Path(path).rglob('*.csv'))
print(f"Found {len(csvs_found)} CSVs in cache: {[f.name for f in csvs_found]}")

# Copy to Drive
for item in csvs_found:
    dest = RAW_DIR / item.name
    if not dest.exists():
        shutil.copy2(item, dest)
        print(f"Copied {item.name} to Drive.")
    else:
        print(f"{item.name} already exists in Drive.")

# Re-run Schema Inspection
print("\n--- Inspecting Amazon CSVs ---")
csv_files = list(RAW_DIR.rglob('*.csv'))
schema_info = {}

for csv_path in csv_files:
    print(f"\nFile: {csv_path.name}")
    try:
        preview = pd.read_csv(csv_path, nrows=5)
        print("Columns:", preview.columns.tolist())
        print("Preview:\n", preview.head(2).to_string())
        schema_info[csv_path.name] = {
            "columns": preview.columns.tolist(),
            "dtypes": preview.dtypes.astype(str).to_dict()
        }
    except Exception as e:
        print(f"Failed to read {csv_path.name}: {repr(e)}")

schema_path = Path('/content/drive/MyDrive/NLP_Lab/processed_data/amazon_reviews/amazon_schema.json')
with open(schema_path, "w") as f:
    json.dump(schema_info, f, indent=2)
print(f"\nAmazon schema saved: {schema_path.exists()}")

--- Applying Fix: Locating Kaggle files ---
Using Colab cache for faster access to the 'amazon-reviews-for-sentianalysis-finegrained-csv' dataset.
Found 2 CSVs in cache: ['train.csv', 'test.csv']
train.csv already exists in Drive.
test.csv already exists in Drive.

--- Inspecting Amazon CSVs ---

File: test.csv
Columns: ['class_index', 'review_title', 'review_text']
Preview:
    class_index             review_title                                                                                                                                                                                                                                                                                                                                                                                                   review_text
0            1          mens ultrasheer  This model may be ok for sedentary types, but I'm active and get around alot in my job - consistently found these stockings rolled up down by 

In [4]:
import pandas as pd
from pathlib import Path
import json

RAW_DIR = Path('/content/drive/MyDrive/NLP_Lab/raw_data/kaggle_amazon')
PROCESSED_DIR = Path('/content/drive/MyDrive/NLP_Lab/processed_data/amazon_reviews')

train_path = RAW_DIR / 'train.csv'
test_path = RAW_DIR / 'test.csv'

print("--- Amazon Dataset Audit & Chunked Processing ---")

def audit_and_save(input_path, split_name, chunksize=500000):
    print(f"\nProcessing {split_name}...")
    total_rows = 0
    null_counts = {'review_title': 0, 'review_text': 0}
    class_counts = {i: 0 for i in range(1, 6)}

    chunks = []
    # Read in chunks to avoid OOM limits
    for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize)):
        total_rows += len(chunk)

        # Count nulls
        null_counts['review_title'] += int(chunk['review_title'].isna().sum())
        null_counts['review_text'] += int(chunk['review_text'].isna().sum())

        # Count classes
        counts = chunk['class_index'].value_counts().to_dict()
        for k, v in counts.items():
            class_counts[k] += int(v)

        # Basic cleaning: fill nulls to prevent Parquet typing issues
        chunk['review_title'] = chunk['review_title'].fillna('')
        chunk['review_text'] = chunk['review_text'].fillna('')

        # Save as Parquet shard
        out_path = PROCESSED_DIR / f"{split_name}_chunk_{i}.parquet"
        chunk.to_parquet(out_path, index=False)
        chunks.append(out_path.name)

    print(f"Total rows: {total_rows}")
    print(f"Nulls: {null_counts}")
    print(f"Class distribution: {class_counts}")
    return total_rows, class_counts, chunks

train_rows, train_dist, train_chunks = audit_and_save(train_path, 'train')
test_rows, test_dist, test_chunks = audit_and_save(test_path, 'test')

audit_report = {
    "train_rows": train_rows,
    "train_class_distribution": train_dist,
    "train_chunks": train_chunks,
    "test_rows": test_rows,
    "test_class_distribution": test_dist,
    "test_chunks": test_chunks
}

with open(PROCESSED_DIR / 'amazon_data_quality.json', 'w') as f:
    json.dump(audit_report, f, indent=2)

print("\nAmazon dataset audit complete and Parquet shards saved.")

--- Amazon Dataset Audit & Chunked Processing ---

Processing train...
Total rows: 3000000
Nulls: {'review_title': 188, 'review_text': 0}
Class distribution: {1: 600000, 2: 600000, 3: 600000, 4: 600000, 5: 600000}

Processing test...
Total rows: 650000
Nulls: {'review_title': 26, 'review_text': 0}
Class distribution: {1: 130000, 2: 130000, 3: 130000, 4: 130000, 5: 130000}

Amazon dataset audit complete and Parquet shards saved.


In [5]:
from datasets import load_dataset
import pandas as pd
import json
from pathlib import Path

print("--- GoEmotions Dataset Audit ---")
emotion_ds = load_dataset("google-research-datasets/go_emotions", "simplified")
labels_list = emotion_ds["train"].features['labels'].feature.names

def audit_goemotions(ds_split, split_name):
    print(f"\nAudit for {split_name}:")
    df = ds_split.to_pandas()

    # Count labels per example (multi-label check)
    df['label_count'] = df['labels'].apply(len)
    print("Labels per example:\n", df['label_count'].value_counts().sort_index().to_dict())

    # Flatten list of labels to count prevalence
    all_labels = [label for sublist in df['labels'] for label in sublist]
    prevalence = pd.Series(all_labels).value_counts().rename(index=dict(enumerate(labels_list)))

    print("Top 3 classes:", prevalence.head(3).to_dict())

    # Convert numpy types to native Python for JSON serialization
    return {
        "total_rows": int(len(df)),
        "labels_per_example_distribution": {int(k): int(v) for k, v in df['label_count'].value_counts().items()},
        "class_prevalence": {str(k): int(v) for k, v in prevalence.items()}
    }

go_audit = {
    "train": audit_goemotions(emotion_ds["train"], "train"),
    "validation": audit_goemotions(emotion_ds["validation"], "validation"),
    "test": audit_goemotions(emotion_ds["test"], "test")
}

PROCESSED_DIR = Path('/content/drive/MyDrive/NLP_Lab/processed_data/go_emotions')
with open(PROCESSED_DIR / 'goemotions_data_quality.json', 'w') as f:
    json.dump(go_audit, f, indent=2)

print("\nGoEmotions audit saved.")

--- GoEmotions Dataset Audit ---

Audit for train:
Labels per example:
 {1: 36308, 2: 6541, 3: 532, 4: 28, 5: 1}
Top 3 classes: {'neutral': 14219, 'admiration': 4130, 'approval': 2939}

Audit for validation:
Labels per example:
 {1: 4548, 2: 809, 3: 62, 4: 7}
Top 3 classes: {'neutral': 1766, 'admiration': 488, 'approval': 397}

Audit for test:
Labels per example:
 {1: 4590, 2: 774, 3: 61, 4: 2}
Top 3 classes: {'neutral': 1787, 'admiration': 504, 'gratitude': 352}

GoEmotions audit saved.


In [6]:
import json
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/NLP_Lab')

expected_files = [
    'reports/environment.json',
    'reports/environment_requirements.txt',
    'processed_data/amazon_reviews/amazon_schema.json',
    'processed_data/amazon_reviews/amazon_data_quality.json',
    'processed_data/go_emotions/goemotions_schema.json',
    'processed_data/go_emotions/goemotions_data_quality.json'
]

print("--- Final Notebook 00 Gates ---")
all_passed = True

for f in expected_files:
    path = ROOT / f
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = 'PASS' if exists and size > 0 else 'FAIL'
    print(f"{path.name:30} | {status} | {size} bytes")
    if status == 'FAIL':
        all_passed = False

if all_passed:
    print("\nSTATUS: PASS. All expected outputs from Notebook 00 are securely saved to Drive.")
else:
    print("\nSTATUS: FAIL. Missing or empty expected outputs. Fix before proceeding.")

--- Final Notebook 00 Gates ---
environment.json               | PASS | 224 bytes
environment_requirements.txt   | PASS | 14223 bytes
amazon_schema.json             | PASS | 435 bytes
amazon_data_quality.json       | PASS | 568 bytes
goemotions_schema.json         | PASS | 537 bytes
goemotions_data_quality.json   | PASS | 2495 bytes

STATUS: PASS. All expected outputs from Notebook 00 are securely saved to Drive.
